# Detecção de garimpo em imagens de satélite da Amazônia com DenseNet121

**Transfer learning e fine-tuning em Redes Neurais Convolucionais**

Machine Learning e Deep Learning · Profa. Polyana Santos Fonseca Nascimento · CESUPA · 2026/2

Classificamos recortes de imagens Sentinel-2 da Amazônia em `sem_garimpo` ou `com_garimpo` com uma DenseNet121 pré-treinada no ImageNet. Comparamos três condições: base congelada, fine-tuning A e fine-tuning B.

## 1. Apresentação

### • **Integrantes**

| Nome | Matrícula |
| --- | --- |
| Gabriel Costa de Miranda | 24070036 |
| João Pedro Silva da Silva | 24070017 |
| Yago Patrick Schnorr Pinto | 24070054 |

### • **Contexto do problema**

Garimpos abertos na floresta deixam marcas visíveis do espaço: clareiras de solo exposto, cavas alagadas e rios com água barrenta. Essas marcas aparecem em imagens de satélite gratuitas, como as do Sentinel-2, que cobrem toda a Amazônia a cada poucos dias.

A área a vigiar, porém, é grande demais para inspeção visual de cada imagem. Um classificador automático pode fazer a primeira triagem e apontar onde vale a pena olhar com atenção.

### • **O que se classifica e finalidade**

Cada amostra é um recorte RGB de 128 × 128 pixels de um mosaico Sentinel-2. Com resolução de 10 m, o recorte cobre cerca de 1,28 km × 1,28 km. O modelo responde a uma pergunta binária: o recorte está em uma área de garimpo (`com_garimpo`) ou não (`sem_garimpo`)?

Uma finalidade possível é a **triagem para fiscalização**: aplicar o modelo a grandes áreas e priorizar, para análise de especialistas ou de órgãos ambientais, os recortes com maior probabilidade de garimpo. Nesse uso, o erro mais grave é o falso negativo, um garimpo que passa sem ser detectado.

### • **Dataset e justificativa da escolha**

Usamos o [Dataset Amazonia Garimpo Binario](https://www.kaggle.com/datasets/leogrupioni/dataset-amazonia-garimpo-binario), publicado no Kaggle por Leonardo Grupioni. A construção do dataset é descrita no artigo de Grupioni, Almeida, Gallois e Ranzini, *Detecção de Garimpo na Amazônia com Visão Computacional e Redes Neurais Convolucionais*, Revista Eletrônica de Iniciação Científica em Computação, 2026, [doi:10.5753/reic.2026.7570](https://doi.org/10.5753/reic.2026.7570).

Justificativa da escolha:

- **Relevância:** trata de um problema ambiental real da Amazônia, com uso prático em monitoramento.
- **Origem confiável e rastreável:** as imagens vêm do Sentinel-2 (nível 2A, com máscara de nuvens) e os rótulos vêm dos polígonos de garimpo da RAISG. Um arquivo de manifesto liga cada recorte às suas coordenadas e à fração de sobreposição com os polígonos.
- **Tamanho e equilíbrio:** cerca de 111 mil imagens, com as duas classes em proporções próximas. Isso atende com folga ao mínimo de 1.000 imagens e 100 por classe e permite trabalhar com uma amostra.
- **Pouco explorado:** é um dataset recente, de 2026, não trabalhado em sala.

Trabalhamos com uma amostra estratificada de 20.000 imagens para manter o tempo de treino viável nas três condições.

## 2. Dados

#### Bibliotecas e configuração

In [1]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
from pathlib import Path
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import kagglehub

SEMENTE = 42
NOMES_CLASSES = ["sem_garimpo", "com_garimpo"]
keras.utils.set_random_seed(SEMENTE)
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.21.0 | Keras: 3.15.1
GPU: []


E0000 00:00:1794756970.429628   85688 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


#### Download dos dados pelo kagglehub

In [ ]:
PASTA_DADOS = Path(kagglehub.dataset_download("leogrupioni/dataset-amazonia-garimpo-binario", output_dir="dados/kaggle"))
PASTA_IMAGENS = PASTA_DADOS / "dataset_amazonia_garimpo_binario"
print(sorted(p.name for p in PASTA_IMAGENS.iterdir()))

['com_garimpo', 'manifesto_chips.csv', 'sem_garimpo']


#### Inspeção dos arquivos

In [3]:
arquivos = pd.DataFrame({"caminho": [p for p in PASTA_IMAGENS.rglob("*") if p.is_file()]})
arquivos["extensao"] = arquivos.caminho.map(lambda p: p.suffix)
arquivos["pasta"] = arquivos.caminho.map(lambda p: p.parent.name)
pd.crosstab(arquivos.pasta, arquivos.extensao, margins=True, margins_name="total")

extensao,.csv,.png,total
pasta,,,
com_garimpo,0,55541,55541
dataset_amazonia_garimpo_binario,1,0,1
sem_garimpo,0,56043,56043
total,1,111584,111585


#### Entendendo o manifesto.csv

In [9]:
manifesto = pd.read_csv(PASTA_IMAGENS / "manifesto_chips.csv")
print(manifesto.shape)
manifesto.label_str.value_counts()

(111584, 27)


label_str
sem_garimpo    56043
com_garimpo    55541
Name: count, dtype: int64

In [5]:
# Colunas que descrevem a origem dos recortes: têm um único valor em todo o dataset.
manifesto[["tile_size_px", "bands_used", "dtype_src", "utm_epsg"]].drop_duplicates()

,tile_size_px,bands_used,dtype_src,utm_epsg
0,128,"['B4', 'B3', 'B2']",float32,32721


In [6]:
# Os recortes vêm de 12 blocos vizinhos do mesmo mosaico.
blocos = manifesto.groupby("tif_source").agg(
    recortes=("label_int", "size"),
    fracao_com_garimpo=("label_int", "mean"),
    latitude=("centroid_lat", "mean"),
    longitude=("centroid_lon", "mean"))
blocos.index = blocos.index.str.extract(r"-(\d+-\d+)\.tif$")[0].rename("bloco")
blocos.round(3)

,recortes,fracao_com_garimpo,latitude,longitude
bloco,,,,
0000000000-0000000000,11236,0.465,-5.105,-57.729
0000000000-0000013568,11236,0.597,-5.105,-56.510
0000000000-0000027136,11130,0.235,-5.105,-55.297
0000013568-0000000000,11236,0.578,-6.324,-57.729
0000013568-0000013568,11236,0.892,-6.324,-56.510
0000013568-0000027136,11130,0.550,-6.324,-55.297
0000027136-0000000000,11236,0.060,-7.543,-57.729
0000027136-0000013568,11236,0.567,-7.543,-56.510
0000027136-0000027136,11130,0.820,-7.543,-55.297


#### Formato das imagens

In [18]:
def ler_png(caminho):
    return tf.io.decode_png(tf.io.read_file(str(caminho))).numpy()

exemplos = manifesto.groupby("label_str").sample(5, random_state=SEMENTE)
pd.DataFrame([{
    "classe": linha.label_str, 
    "formato": (imagem := ler_png(PASTA_IMAGENS / linha.png_path)).shape,
    "tipo": imagem.dtype.name, 
    "mínimo": imagem.min(), 
    "máximo": imagem.max(), 
    "média": round(imagem.mean(), 1)
    } for linha in exemplos.itertuples()])

,classe,formato,tipo,mínimo,máximo,média
0,com_garimpo,"(128, 128, 3)",uint8,0,194,7.9
1,com_garimpo,"(128, 128, 3)",uint8,0,97,5.2
2,com_garimpo,"(128, 128, 3)",uint8,0,34,4.8
3,com_garimpo,"(128, 128, 3)",uint8,0,218,13.8
4,com_garimpo,"(128, 128, 3)",uint8,0,45,8.1
5,sem_garimpo,"(128, 128, 3)",uint8,0,63,9.0
6,sem_garimpo,"(128, 128, 3)",uint8,0,102,6.6
7,sem_garimpo,"(128, 128, 3)",uint8,0,44,7.0
8,sem_garimpo,"(128, 128, 3)",uint8,0,37,6.8
9,sem_garimpo,"(128, 128, 3)",uint8,0,40,7.1


#### Origem dos rótulos
O rótulo vem da sobreposição de cada recorte com polígonos de garimpo da RAISG (`overlap_ratio`).

In [16]:
manifesto.groupby("label_str")['overlap_ratio'].describe()

,count,mean,std,min,25%,50%,75%,max
label_str,,,,,,,,
com_garimpo,55541.0,1.003981,0.274265,0.050329,0.999998,1.0,1.0,3.310692
sem_garimpo,56043.0,0.000227,0.002678,0.000000,0.000000,0.0,0.0,0.049813


#### **Resumo da inspeção**

- **Formato:** PNG RGB de 8 bits, 128 × 128 pixels, bandas B4, B3 e B2 do Sentinel-2. Não é preciso converter.
- **Classes:** 56.043 `sem_garimpo` e 55.541 `com_garimpo` (50,2% e 49,8%). Não usaremos `class_weight`.
- **Brilho:** as imagens são escuras (média perto de 8 em 255). Tratamos isso no pré-processamento.
- **Rótulo:** `com_garimpo` quando mais de 5% do recorte cai em um polígono da RAISG. Os polígonos marcam áreas de garimpo, não só o solo exposto; um recorte positivo pode ser quase todo floresta. Verificamos isso visualmente na análise exploratória.
- **Região:** 12 blocos vizinhos, com fração de garimpo de 0% a 89%. As coordenadas permitem dividir os dados por região.